In [2]:
import os
import time
import pandas as pd
import numpy as np

# 1. Definir rutas
path_csv = "../data/raw/operaciones-aeropuertos.csv"
path_parquet = "../data/raw/bitacora-vuelos.parquet"

# 2. Evaluación del archivo Parquet 
start_time = time.time()
df = pd.read_parquet(path_parquet)
time_parquet = time.time() - start_time

mem_parquet = df.memory_usage(deep=True).sum() / (1024**2) # MB
size_parquet = os.path.getsize(path_parquet) / (1024**2)  # MB

print("--- PERFILAMIENTO TÉCNICO PARQUET ---")
print(f"Dimensiones (Shape) : {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Tamaño en disco    : {size_parquet:.2f} MB")
print(f"Tiempo de carga    : {time_parquet:.4f} segundos")
print(f"Uso de memoria RAM : {mem_parquet:.2f} MB")
print("\n--- PRIMEROS REGISTROS ---")
df.head(3)

--- PERFILAMIENTO TÉCNICO PARQUET ---
Dimensiones (Shape) : 11,163,791 filas x 12 columnas
Tamaño en disco    : 131.84 MB
Tiempo de carga    : 6.9549 segundos
Uso de memoria RAM : 2529.24 MB

--- PRIMEROS REGISTROS ---


,dt_operacion,aeropuerto_oaci,tipo_operacion,aeropuerto_dgac_orig_dest,aerolinea_dgac,numero_vuelo,actividad_cod,matricula,modelo_avion,modelo_avion_desc,es_internacional,pmd
0,1999-12-31 21:00:00-03:00,SCEL,A,SPIM,LAN,631,U,CCCDM,B763,767-352ER,True,187.00000
1,1999-12-31 21:00:00-03:00,SCTC,D,SCUG,6,0,P,CCSPC,C172,172B,False,1.00000
2,1999-12-31 21:04:00.001000-03:00,SCEL,A,SCTC,LAN,4248,U,CCCHS,EC35,EC-135 T2,False,2.84000


In [3]:
# Inspección de tipos e integridad
print("--- TIPOS DE DATOS Y VALORES AUSENTES ---")
perfilamiento = pd.DataFrame({
    'Tipo_Dato': df.dtypes,
    'Valores_Nulos': df.isna().sum(),
    'Porcentaje_Nulos (%)': (df.isna().sum() / len(df) * 100).round(2),
    'Valores_Unicos': df.nunique()
})
perfilamiento

--- TIPOS DE DATOS Y VALORES AUSENTES ---


,Tipo_Dato,Valores_Nulos,Porcentaje_Nulos (%),Valores_Unicos
dt_operacion,"datetime64[us, America/Santiago]",0,0.00,6527475
aeropuerto_oaci,str,0,0.00,69
tipo_operacion,str,0,0.00,4
aeropuerto_dgac_orig_dest,str,0,0.00,1671
aerolinea_dgac,str,0,0.00,2568
numero_vuelo,str,482145,4.32,7442
actividad_cod,str,0,0.00,26
matricula,str,0,0.00,13835
modelo_avion,str,0,0.00,1286
modelo_avion_desc,str,0,0.00,2325


In [4]:
# --- PREPARACIÓN E INGENIERÍA DE VARIABLES ---

df_proc = df.copy()

# 1. Normalización de cadenas y tipos
df_proc['aerolinea_dgac'] = df_proc['aerolinea_dgac'].astype(str).str.strip().str.upper()
df_proc['aeropuerto_oaci'] = df_proc['aeropuerto_oaci'].astype(str).str.strip().str.upper()

# 2. Conversión de PMD a numérico (trata errores/caracteres como NaN)
if 'pmd' in df_proc.columns:
    df_proc['pmd'] = pd.to_numeric(df_proc['pmd'], errors='coerce')

# 3. Tratamiento justificado de nulos en número de vuelo
# Los vuelos sin número corresponden principalmente a aviación general/privada (no comercial)
df_proc['numero_vuelo'] = df_proc['numero_vuelo'].fillna('SIN_NUMERO')

# 4. INGENIERÍA DE VARIABLES (3 Variables Derivadas)
# V1: Año de la operación
df_proc['anio'] = df_proc['dt_operacion'].dt.year

# V2: Mes de la operación (1-12)
df_proc['mes'] = df_proc['dt_operacion'].dt.month

# V3: Trimestre / Quarter (Q1-Q4)
df_proc['trimestre'] = df_proc['dt_operacion'].dt.quarter

# 5. Guardar el dataset procesado 
output_path = "../data/processed/transportes_procesado.parquet"
df_proc.to_parquet(output_path, index=False)

print("--- PROCESAMIENTO FINALIZADO ---")
print(f"Dataset procesado guardado en: {output_path}")
print(f"Dimensiones finales: {df_proc.shape[0]:,} filas x {df_proc.shape[1]} columnas")
df_proc[['dt_operacion', 'anio', 'mes', 'trimestre', 'aerolinea_dgac', 'es_internacional']].head(3)

--- PROCESAMIENTO FINALIZADO ---
Dataset procesado guardado en: ../data/processed/transportes_procesado.parquet
Dimensiones finales: 11,163,791 filas x 15 columnas


,dt_operacion,anio,mes,trimestre,aerolinea_dgac,es_internacional
0,1999-12-31 21:00:00-03:00,1999,12,4,LAN,True
1,1999-12-31 21:00:00-03:00,1999,12,4,6,False
2,1999-12-31 21:04:00.001000-03:00,1999,12,4,LAN,False
